# MOSS-Image on Kaggle (ComfyUI)

部署 ComfyUI（最新 master，含 MOSS-Image 原生支持）+ Moody Moss Mix V3.3（MOSS-Image 架构 INT8 checkpoint，从账号私有数据集 moody-moss-mix-v33 挂载读取，不再从 HuggingFace 下载）+ Qwen3.5-9B w4a8 + FLUX.2 small VAE，跑一张竖版 3:4 林间人像测试图。KSampler 25 步、CFG 3.5（作者推荐值）。

模型来自 `Comfy-Org/MOSS-Image`（公开仓库，Apache 2.0，无需 token）。测试提示词沿用之前在 SDXL 上被 CLIP 截断的那段长提示词，正好验证 MOSS-Image 约 4K token 的提示词容量；提示词增强器不开，提示词原样送入模型。

注意：ComfyUI 服务由出图那格用 subprocess.Popen 在本进程内启动（Kaggle 版本运行里跨格的 nohup 后台进程活不下来），跑完即停。

LoRA（2026-11-15）：注册表 MOSS_LORAS + 选择列表 LORA_NAMES，串联 LoraLoaderModelOnly 节点（只改模型，不动 CLIP）；观澜（HF，0.8）→ Blender3D（Kaggle 数据集，1.0）固定顺序，默认只挂观澜，空列表 = 不挂 LoRA。
SeedVR 放大（2026-11-15）：底图 1024x1536（2:3，约 1.57MP）生成后，用 SeedVR2 7B（DiT 7.88GB + VAE 0.47GB，私有数据集 kitzhang2019/seedvr2-7b 挂载，软链接进 ComfyUI/models/SEEDVR2/，不占 /kaggle/working 磁盘）放大到 2048x3072（约 6.3MP，resolution 2048、batch 5、blocks_to_swap 36、sdpa、color lab，T4 上约 390 秒；4096 会在 DiT 注意力层 OOM）。出图格分两段跑并分别计时；成品另存 WebP 精细模式（q95 起步，20MB 上限）——2048x3072 的 PNG 约 8.8MB、超站回存 8MB 上限，站回存走 WebP。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working
!python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
!if [ -d /kaggle/working/ComfyUI ]; then echo comfyui-exists; else git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI; fi
!cd /kaggle/working/ComfyUI && grep -v -E "^(torch|torchvision|torchaudio)(==|>=|~=|>|<| |$)" requirements.txt > /tmp/comfy_reqs.txt && pip install -q -r /tmp/comfy_reqs.txt
!pip install -q torchsde
!python3 -c "import torchsde; print('torchsde ok')"
!ls /kaggle/working/ComfyUI | head -20

# SeedVR2 plugin (numz, 2026-11-15): the SeedVR2VideoUpscaler node upscales the
# low-res generation to 2048x3072 in phase 2. Same install as the verified 7B test.
!if [ -d /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler ]; then echo seedvr-plugin-exists; else git clone --depth 1 https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler; fi
!cd /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler && grep -v -E "^(torch|torchvision)(==|>=|~=|>|<| |$)" requirements.txt > /tmp/seedvr_reqs.txt && pip install -q -r /tmp/seedvr_reqs.txt
!ls /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler | head -20

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
from huggingface_hub import hf_hub_download
base = "/kaggle/working/ComfyUI/models"
# NOTE: filenames must include the repo subdirectory; hf_hub_download places the file at local_dir/filename
#
# LoRA registry (2026-11-15): two MOSS LoRAs, each from its own source.
#   guanlan:   KOOK-MOSS-SFT-观澜.safetensors, 0.8, HF repo RunningHubAI/rh-kook-moss-sft-lora
#   blender3d: Moss_Blender3D_V2.safetensors, 1.0, Kaggle dataset kitzhang2019/moss-blender3d-lora-v2
# LORA_NAMES is the per-job selection: the station prepare / App patchNotebook
# overwrite this list from the form selection (default: 观澜 only; empty list =
# no LoRA). Order is fixed: 观澜 first, Blender3D second. LORA_NAME/LORA_STRENGTH
# keep the first entry for the node-presence check and the single-LoRA fallback
# in the run cell. LORA_REPO is kept only so older prepare code does not break.
MOSS_LORAS = {
    "guanlan": ("KOOK-MOSS-SFT-观澜.safetensors", 0.8, "hf:RunningHubAI/rh-kook-moss-sft-lora"),
    "blender3d": ("Moss_Blender3D_V2.safetensors", 1.0, "kitzhang2019/moss-blender3d-lora-v2"),
}
LORA_NAMES = [MOSS_LORAS[k] for k in ("guanlan",)]
LORA_NAME = LORA_NAMES[0][0] if LORA_NAMES else ""
LORA_REPO = "RunningHubAI/rh-kook-moss-sft-lora"
LORA_STRENGTH = LORA_NAMES[0][1] if LORA_NAMES else 0.8
jobs = [
    "text_encoders/qwen3.5_9b_w4a8_gptq.safetensors",
    "vae/flux2-vae_small.safetensors",
]
for fname in jobs:
    p = hf_hub_download(repo_id="Comfy-Org/MOSS-Image", filename=fname, local_dir=base)
    print(fname, round(os.path.getsize(p) / 1e9, 2), "GB")
# Main diffusion model (2026-11-12): Moody Moss Mix V3.3, an INT8
# MOSS-Image checkpoint hosted on Civitai. It is NOT downloaded from
# HuggingFace anymore: it reaches Kaggle as the private dataset
# moody-moss-mix-v33 attached to this kernel, and is located by a
# recursive search under /kaggle/input (the real mount path varies,
# never hard-code it), then copied into ComfyUI's diffusion_models.
import glob, shutil
MOODY_NAME = "Moody-Moss-Mix-V3.3_00001__clean-int8_convrot.safetensors"
hits = sorted(glob.glob("/kaggle/input/**/" + MOODY_NAME, recursive=True))
if not hits:
    raise SystemExit("moody model not found under /kaggle/input: dataset moody-moss-mix-v33 is not attached to this kernel")
moody_dst = os.path.join(base, "diffusion_models", MOODY_NAME)
os.makedirs(os.path.dirname(moody_dst), exist_ok=True)
shutil.copyfile(hits[0], moody_dst)
print("moody model:", hits[0], "->", moody_dst, round(os.path.getsize(moody_dst) / 1e9, 2), "GB")
if LORA_NAMES:
    # Each LoRA comes from its own source: "hf:<repo>" downloads from Hugging Face,
    # otherwise the file is located by recursive search under /kaggle/input (private
    # dataset attached to this kernel) and copied into ComfyUI/models/loras/.
    import glob as _lora_glob
    for _lname, _lstr, _lsrc in LORA_NAMES:
        if _lsrc.startswith("hf:"):
            p = hf_hub_download(repo_id=_lsrc[3:], filename=_lname, local_dir=base + "/loras")
            print("lora", _lname, round(os.path.getsize(p) / 1e9, 2), "GB")
        else:
            _hits = sorted(_lora_glob.glob("/kaggle/input/**/" + _lname, recursive=True))
            if not _hits:
                raise SystemExit("lora file not found under /kaggle/input: " + _lname + " (dataset " + _lsrc + " is not attached to this kernel)")
            _ldest = os.path.join(base, "loras", _lname)
            os.makedirs(os.path.dirname(_ldest), exist_ok=True)
            shutil.copyfile(_hits[0], _ldest)
            print("lora", _lname, "strength", _lstr, round(os.path.getsize(_ldest) / 1e9, 2), "GB", "from", _hits[0])

# SeedVR2 7B (2026-11-15): DiT seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors
# (7.88 GB, AInVFX/SeedVR2_comfyUI) + VAE ema_vae_fp16.safetensors (0.47 GB,
# numz/SeedVR2_comfyUI) come from the private dataset kitzhang2019/seedvr2-7b
# attached to this kernel, located by recursive search under /kaggle/input.
# They are SYMLINKED (not copied) into ComfyUI/models/SEEDVR2/: copying all
# 8.35 GB would overflow /kaggle/working (Qwen already uses ~17 GB of 20 GB).
# Symlinks read through to /kaggle/input, which stays mounted for the run.
import glob as _seedvr_glob
SEEDVR_DIT = "seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors"
SEEDVR_VAE = "ema_vae_fp16.safetensors"
seedvr_dir = os.path.join(base, "SEEDVR2")
os.makedirs(seedvr_dir, exist_ok=True)
for _name in (SEEDVR_DIT, SEEDVR_VAE):
    _hits = sorted(_seedvr_glob.glob("/kaggle/input/**/" + _name, recursive=True))
    if not _hits:
        raise SystemExit("seedvr file not found under /kaggle/input: " + _name + " (dataset kitzhang2019/seedvr2-7b is not attached to this kernel)")
    _link = os.path.join(seedvr_dir, _name)
    if os.path.islink(_link) or os.path.exists(_link):
        os.remove(_link)
    os.symlink(_hits[0], _link)
    print("seedvr:", _hits[0], "->", _link, round(os.path.getsize(_link) / 1e9, 2), "GB")
# Guard: the two files must be visible exactly where the loaders look.
for _name in (SEEDVR_DIT, SEEDVR_VAE):
    _fp = os.path.join(seedvr_dir, _name)
    print("SEEDVR2/", _name, "exists:", os.path.exists(_fp))
    assert os.path.exists(_fp), "missing seedvr file: " + _fp
print("seedvr models ready")
print("all models downloaded")

In [ ]:
import json, time, requests, os, subprocess, sys

BASE = "http://127.0.0.1:8188"
LOG = "/kaggle/working/comfy_server.log"

# Start the ComfyUI server as a child of this cell's python process (nohup across cells does not survive Kaggle version runs).
logf = open(LOG, "wb")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/kaggle/working/ComfyUI",
    stdout=logf, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("server pid:", proc.pid)

up = False
for i in range(36):
    if proc.poll() is not None:
        print("server exited early, code:", proc.returncode)
        break
    try:
        r = requests.get(BASE + "/system_stats", timeout=5)
        if r.status_code == 200:
            up = True
            print("server up after", (i + 1) * 5, "s")
            break
    except Exception:
        pass
    time.sleep(5)
if not up:
    print("server did not come up; log tail:")
    print(open(LOG, errors="ignore").read()[-3000:])
    proc.terminate()
    raise SystemExit("comfyui server failed to start")

PROMPT = """清透柔软的成年面容，配合轻薄低领穿搭的私密时尚写真。绝美23岁东亚美女，身材匀称丰满，胸部明显自然丰满，视觉约E杯，具有自然重量感与柔和圆润轮廓，胸腰差明显但自然，纤细腰部与丰满胸部形成清晰比例，臀腿曲线顺滑圆润饱满。自然散落的黑长发拨在背后，底妆细腻，眉眼舒展，红润唇色带一点柔光；脸微偏向镜头，目光柔软而专注。
她穿香槟色薄绉纱短衬衣，前襟低开成深V，宽松柔软的领边自然落向两侧，深V两侧胸片采用同色不透明薄缎，肩后与短袖保留透光，前摆在腰线上方打一个小结。搭配黑色低腰斜裁微迷你裙，薄缎贴合腰髋，裙摆刚过臀下缘，臀部覆盖完整。她斜倚在窗边窄柜旁，一手撑住身后柜沿，肩颈放松，上身轻转向镜头，近侧腿向前半步，露出的腰线与裙身比例鲜明。
4:5竖幅，50mm取景至大腿中段。侧窗柔光让眼睛、肤色和绉纱边缘清楚透亮，远侧身体保留阴影，背景简洁柔暗。衣料细褶、透光短袖与自然肤理是质感重点。"""
NEGATIVE = "nsfw, worst quality, bad anatomy, watermark, jpeg artifact, blurry, low quality, distorted, deformed, extra limbs, text, lettering, wide landscape, decorative flowers, fantasy styling, cold porcelain skin, excessive skin smoothing, studio light, exaggerated anatomy, visible intimate anatomy"

info = requests.get(BASE + "/object_info", timeout=120).json()
have = set(info.keys())
# LoRA settings are defined in the download cell; default to "no LoRA" if that cell was skipped.
LORA_NAME = globals().get("LORA_NAME", "")
LORA_STRENGTH = globals().get("LORA_STRENGTH", 0.8)
LORA_NAMES = globals().get("LORA_NAMES", None)
need = ["UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode", "EmptyFlux2LatentImage", "KSampler", "VAEDecode", "SaveImage", "LoadImage", "SeedVR2LoadDiTModel", "SeedVR2LoadVAEModel", "SeedVR2VideoUpscaler"]
if LORA_NAME:
    need.append("LoraLoaderModelOnly")
missing = [n for n in need if n not in have]
print("missing nodes:", missing if missing else "none")
if missing:
    proc.terminate()
    raise SystemExit("missing nodes: " + ", ".join(missing))

def node_inputs(class_type, preferred):
    spec = info[class_type]["input"]
    allowed = set(spec.get("required", {})) | set(spec.get("optional", {}))
    return {k: v for k, v in preferred.items() if k in allowed}

use_slg = "SkipLayerGuidanceDiTSimple" in have
print("SLG available:", use_slg)

nodes = {}
nodes["1"] = {"class_type": "UNETLoader", "inputs": node_inputs("UNETLoader", {"unet_name": "Moody-Moss-Mix-V3.3_00001__clean-int8_convrot.safetensors", "weight_dtype": "default"})}
nodes["2"] = {"class_type": "CLIPLoader", "inputs": node_inputs("CLIPLoader", {"clip_name": "qwen3.5_9b_w4a8_gptq.safetensors", "type": "moss_image", "device": "default"})}
nodes["3"] = {"class_type": "VAELoader", "inputs": node_inputs("VAELoader", {"vae_name": "flux2-vae_small.safetensors"})}
nodes["4"] = {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT, "clip": ["2", 0]}}
nodes["5"] = {"class_type": "CLIPTextEncode", "inputs": {"text": NEGATIVE, "clip": ["2", 0]}}
nodes["6"] = {"class_type": "EmptyFlux2LatentImage", "inputs": node_inputs("EmptyFlux2LatentImage", {"width": 1024, "height": 1536, "batch_size": 1})}
print("latent node spec:", json.dumps(info.get("EmptyFlux2LatentImage", {}).get("input", {}))[:600])
print("latent node built:", json.dumps(nodes["6"]))
model_link = ["1", 0]
# LoRA chain: one LoraLoaderModelOnly per entry of LORA_NAMES, in order (model-only,
# no CLIP). Empty list / no LORA_NAME = model goes straight to SLG/KSampler.
_lora_chain = LORA_NAMES
if _lora_chain is None and LORA_NAME:
    _lora_chain = [(LORA_NAME, LORA_STRENGTH, "")]
if _lora_chain:
    for _i, (_lname, _lstr, _lds) in enumerate(_lora_chain):
        _nid = str(20 + _i)
        nodes[_nid] = {"class_type": "LoraLoaderModelOnly", "inputs": node_inputs("LoraLoaderModelOnly", {"model": model_link, "lora_name": _lname, "strength_model": _lstr})}
        model_link = [_nid, 0]
if use_slg:
    nodes["8"] = {"class_type": "SkipLayerGuidanceDiTSimple", "inputs": node_inputs("SkipLayerGuidanceDiTSimple", {"model": model_link, "single_layers": "22", "double_layers": "", "scale": 3.0, "start_percent": 0.01, "end_percent": 1.0})}
    model_link = ["8", 0]
nodes["7"] = {"class_type": "KSampler", "inputs": {"model": model_link, "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0], "seed": 42, "steps": 25, "cfg": 3.5, "sampler_name": "er_sde", "scheduler": "sgm_uniform", "denoise": 1.0}}
nodes["9"] = {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}}
nodes["10"] = {"class_type": "SaveImage", "inputs": {"images": ["9", 0], "filename_prefix": "moss_lowres"}}

# Pre-check: every required input of every node must be present, or fail here with the full list (do not let the server reject it one input at a time).
problems = []
for nid, node in nodes.items():
    req = info[node["class_type"]]["input"].get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems.append(nid + ":" + node["class_type"] + "." + name)
print("missing required inputs:", problems if problems else "none")
if problems:
    proc.terminate()
    raise SystemExit("missing required inputs: " + ", ".join(problems))

# ---------- SeedVR helpers (from the verified 7B test notebook) ----------
def combo_opts_for_cfg(cfg):
    """Options list for a combo input spec, or None if the input is not a combo.
    Old format: [["opt1", "opt2"], {...}]. New format (current ComfyUI master /
    SeedVR2 plugin): ["COMBO", {"options": [...], "default": ..., ...}]."""
    if not isinstance(cfg, list) or not cfg:
        return None
    if isinstance(cfg[0], list):
        return cfg[0]
    if cfg[0] == "COMBO" and len(cfg) > 1 and isinstance(cfg[1], dict) and "options" in cfg[1]:
        return cfg[1]["options"]
    return None

def combo_options(class_type, name):
    """Return the option list for a combo input (both spec formats)."""
    spec = info[class_type]["input"]
    cfg = spec.get("required", {}).get(name) or spec.get("optional", {}).get(name)
    opts = combo_opts_for_cfg(cfg)
    return opts if opts is not None else ([cfg[0]] if cfg else [])

# Print the SeedVR2 node specs so a wrong input name/value is caught here, not after a long run.
for ct in ["SeedVR2LoadDiTModel", "SeedVR2LoadVAEModel", "SeedVR2VideoUpscaler"]:
    print(ct, "spec:", json.dumps(info[ct]["input"])[:1200])

# Resolve device + model values from the live specs (never hard-code a combo value).
dit_device = combo_options("SeedVR2LoadDiTModel", "device")[0]
vae_device = combo_options("SeedVR2LoadVAEModel", "device")[0]
dit_models = combo_options("SeedVR2LoadDiTModel", "model")
vae_models = combo_options("SeedVR2LoadVAEModel", "model")
print("dit device:", dit_device, "| vae device:", vae_device)
print("dit models available:", dit_models)
print("vae models available:", vae_models)
DIT_MODEL = "seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors"
VAE_MODEL = "ema_vae_fp16.safetensors"
assert DIT_MODEL in dit_models, "DiT model not in loader options: " + DIT_MODEL
assert VAE_MODEL in vae_models, "VAE model not in loader options: " + VAE_MODEL

# ---------- Phase 2 nodes: SeedVR2 7B upscale (1024x1536 -> 2048x3072) ----------
# Verified 2026-11-15 (kernel kitzhang2019/seedvr-7b-ddimbeta-test-20261114):
# resolution 2048, max 6000, batch 5, blocks_to_swap 36, VAE tiled 1024/128,
# sdpa attention, color lab. 390 s on T4; 4096 OOMs in DiT attention.
nodes_seedvr = {}
nodes_seedvr["31"] = {"class_type": "LoadImage", "inputs": node_inputs("LoadImage", {"image": "moss_lowres.png"})}
nodes_seedvr["32"] = {"class_type": "SeedVR2LoadDiTModel", "inputs": node_inputs("SeedVR2LoadDiTModel", {
    "model": DIT_MODEL, "device": dit_device, "blocks_to_swap": 36,
    "swap_io_components": False, "offload_device": "cpu", "attention_mode": "sdpa"})}
nodes_seedvr["33"] = {"class_type": "SeedVR2LoadVAEModel", "inputs": node_inputs("SeedVR2LoadVAEModel", {
    "model": VAE_MODEL, "device": vae_device, "encode_tiled": True, "encode_tile_size": 1024,
    "encode_tile_overlap": 128, "decode_tiled": True, "decode_tile_size": 1024,
    "decode_tile_overlap": 128, "tile_debug": "false", "offload_device": "cpu"})}
nodes_seedvr["34"] = {"class_type": "SeedVR2VideoUpscaler", "inputs": node_inputs("SeedVR2VideoUpscaler", {
    "image": ["31", 0], "dit": ["32", 0], "vae": ["33", 0], "seed": 42,
    "resolution": 2048, "max_resolution": 6000, "batch_size": 5,
    "uniform_batch_size": False, "temporal_overlap": 0, "prepend_frames": 0,
    "color_correction": "lab", "input_noise_scale": 0.0})}
nodes_seedvr["35"] = {"class_type": "SaveImage", "inputs": {"images": ["34", 0], "filename_prefix": "moss_fashion"}}


def submit_and_wait(nodes, save_node_id, out_path, timeout_s, label):
    """Submit one prompt, poll until success/error/timeout, download the SaveImage output. Returns (done, elapsed_s)."""
    r = requests.post(BASE + "/prompt", json={"prompt": nodes}, timeout=180)
    print(label, "submit:", r.status_code, r.text[:400])
    r.raise_for_status()
    pid = r.json()["prompt_id"]
    print(label, "prompt_id:", pid)
    t0 = time.time()
    done = False
    while time.time() - t0 < timeout_s:
        h = requests.get(BASE + "/history/" + pid, timeout=60).json()
        if pid in h:
            st = h[pid].get("status", {})
            sstr = st.get("status_str")
            if sstr == "success":
                outs = h[pid]["outputs"]
                imgs = outs.get(save_node_id, {}).get("images", [])
                print(label, "images:", imgs)
                if imgs:
                    im = imgs[0]
                    q = {"filename": im["filename"], "subfolder": im.get("subfolder", ""), "type": im.get("type", "output")}
                    data = requests.get(BASE + "/view", params=q, timeout=300).content
                    open(out_path, "wb").write(data)
                    print(label, "saved", out_path, len(data), "bytes")
                    done = True
                break
            if sstr == "error":
                print(label, "RUN ERROR:", json.dumps(h[pid], indent=2)[:3000])
                break
        time.sleep(15)
    elapsed = round(time.time() - t0)
    print(label, "done:", done, "elapsed:", elapsed, "s")
    return done, elapsed

# Phase 1: low-res generation (1024x1536). MOSS 25 steps, expect ~20 min; allow 55 min.
gen_ok, gen_s = submit_and_wait(nodes, "10", "/kaggle/working/moss_lowres.png", 3300, "GEN")
if not gen_ok:
    proc.terminate()
    raise SystemExit("phase 1 (generation) failed")

# LoadImage reads from ComfyUI/input/; copy the low-res output there.
import shutil
os.makedirs("/kaggle/working/ComfyUI/input", exist_ok=True)
shutil.copyfile("/kaggle/working/moss_lowres.png", "/kaggle/working/ComfyUI/input/moss_lowres.png")
print("copied low-res to ComfyUI/input/moss_lowres.png")

# Re-fetch object_info now that the low-res file is in ComfyUI/input/:
# the LoadImage combo options are snapshotted from the input dir at fetch
# time, so the pre-check must see the file Phase 1 just produced.
info = requests.get(BASE + "/object_info", timeout=120).json()

# Pre-check for the SeedVR nodes: every required input present, and every
# combo value one of the live options (same check as the generation phase).
problems_sv = []
for nid, node in nodes_seedvr.items():
    spec = info[node["class_type"]]["input"]
    req = spec.get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems_sv.append(nid + ":" + node["class_type"] + "." + name)
    for name, val in node["inputs"].items():
        cfg = req.get(name) or spec.get("optional", {}).get(name)
        opts = combo_opts_for_cfg(cfg)
        if opts is not None and isinstance(val, (str, int, float)):
            if val not in opts:
                problems_sv.append(nid + ":" + node["class_type"] + "." + name + " value '" + str(val) + "' not in " + str(opts))
print("seedvr missing required inputs:", problems_sv if problems_sv else "none")
if problems_sv:
    proc.terminate()
    raise SystemExit("seedvr missing required inputs: " + ", ".join(problems_sv))

# Phase 2: SeedVR2 7B upscale (1024x1536 -> 2048x3072). Verified ~390 s; allow 30 min.
sv_ok, sv_s = submit_and_wait(nodes_seedvr, "35", "/kaggle/working/moss_fashion.png", 1800, "SEEDVR")
if not sv_ok:
    proc.terminate()
    raise SystemExit("phase 2 (SeedVR upscale) failed")

# Phase 3: WebP fine export (quality 95 / method 6; step down to 90, then 85 only
# if still over 20MB). The 2048x3072 PNG is ~8.8MB, over the station's 8MB
# return cap, so the WebP is the file the station fetches back.
from PIL import Image as _WImage
def webp_fine(png_path, webp_path):
    sz = None
    q = None
    for _q in (95, 90, 85):
        _WImage.open(png_path).convert("RGB").save(webp_path, "WEBP", quality=_q, method=6)
        _sz = os.path.getsize(webp_path)
        print("webp q%d" % _q, webp_path, _sz, "bytes")
        sz, q = _sz, _q
        if _sz <= 20 * 1024 * 1024 or _q == 85:
            break
    return sz, q

webp_s, webp_q = webp_fine("/kaggle/working/moss_fashion.png", "/kaggle/working/moss_fashion.webp")

proc.terminate()
print("server stopped")
print("RESULT gen_s=%d seedvr_s=%d webp_bytes=%d webp_q=%d" % (gen_s, sv_s, webp_s, webp_q))
print("exists:", os.path.exists("/kaggle/working/moss_fashion.webp"), "bytes:", os.path.getsize("/kaggle/working/moss_fashion.webp") if os.path.exists("/kaggle/working/moss_fashion.webp") else 0)
assert os.path.exists("/kaggle/working/moss_fashion.webp"), "no output webp: /kaggle/working/moss_fashion.webp"


In [ ]:
from IPython.display import Image, display
import os
# NOTE: do NOT display the .webp here -- IPython cannot embed webp
# ("Cannot embed the 'webp' image format") and papermill marks the whole
# kernel ERROR even though generation succeeded. Only the PNG is displayed.
for p in ["/kaggle/working/moss_fashion.png", "/kaggle/working/moss_fashion.webp"]:
    print(p, "exists:", os.path.exists(p), "bytes:", os.path.getsize(p) if os.path.exists(p) else 0)
if os.path.exists("/kaggle/working/moss_fashion.png"):
    display(Image(filename="/kaggle/working/moss_fashion.png"))
